# HI-VIS — Automated PPE Compliance Detection

**Le Wagon Data Science & AI · Batch #2303 · Final Project**

Point a model at a construction-site photograph and get back who is in it, what PPE they're wearing, and what's missing — turning site photos every project already takes into a checked, dated, searchable safety record.

This notebook is the shared starting point: data loading, EDA, a CNN baseline, and a YOLO object-detection model.


## 1 · Setup

In [4]:
# Core
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Modelling (transfer learning baseline)
# import tensorflow as tf
# from tensorflow.keras import layers, Sequential
# from tensorflow.keras.applications.vgg16 import VGG16

# Object detection
# from ultralytics import YOLO   # pip install ultralytics

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## 2 · Data

**Primary dataset:** [Construction Site Safety Image Dataset (Roboflow export)](https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow) — 2,801 images, YOLOv8 format, CC BY 4.0.

Classes (10): `Hardhat`, `Mask`, `NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`, `Person`, `Safety Cone`, `Safety Vest`, `machinery`, `vehicle`.

Reference notebook using this same dataset: [PPE Kit Detection — Construction Site Safety](https://www.kaggle.com/code/sajjadalishah/ppe-kit-detection-construction-site-safety).



### ⚠️ If you downloaded the reference notebook's "Output" from Kaggle

Kaggle lets you download a public notebook's entire output bundle, not just the input dataset — put in all in `data/`, it unpacks into **three unrelated things**, not one clean dataset:

```
data/
├── css-data/                          ← the ACTUAL dataset — use this as DATA_DIR
│   ├── README.dataset.txt / README.roboflow.txt
│   ├── train/{images,labels}/
│   ├── valid/{images,labels}/
│   └── test/{images,labels}/
├── results_yolov8n_100e/              ← the reference notebook's OWN completed training run
│   └── kaggle/working/
│       ├── __notebook__.ipynb         ← the reference notebook's actual source + outputs
│       ├── yolov8n.pt                 ← base COCO checkpoint (untouched)
│       ├── ppe_data.yaml              ← the data.yaml that run was trained with
│       └── runs/detect/train/weights/
│           ├── best.pt                ← already fine-tuned on THIS dataset (100 epochs)
│           └── last.pt
└── source_files/source_files/         ← random demo images/videos, not training data
```


In [5]:
DATA_DIR = Path("data/css-data")

for split in ["train", "valid", "test"]:
    img_dir = DATA_DIR / split / "images"
    lbl_dir = DATA_DIR / split / "labels"
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*"))) if lbl_dir.exists() else 0
    print(f"{split:6s} images={n_img:5d}  labels={n_lbl:5d}")


train  images=    0  labels=    0
valid  images=    0  labels=    0
test   images=    0  labels=    0


In [ ]:
# The Roboflow export doesn't always ship its own data.yaml (ours didn't) — write one.
# Class order below matches ppe_data.yaml from the reference notebook's run, so it's
# consistent with results_yolov8n_100e's weights too.
import yaml


# TODO: fix the path to point it at your downloaded data folder

CLASS_NAMES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]

data_cfg = {
    "train": str((DATA_DIR / "train" / "images").resolve()),
    "val": str((DATA_DIR / "valid" / "images").resolve()),
    "test": str((DATA_DIR / "test" / "images").resolve()),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATA_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(f"wrote {yaml_path}")
print(data_cfg)


FileNotFoundError: [Errno 2] No such file or directory: 'data/css-data/data.yaml'

### Quick sanity check — try the reference checkpoint before training anything

`results_yolov8n_100e` contains a YOLOv8n **already fine-tuned on this exact dataset** (100 epochs, on a Kaggle P100, per the reference notebook). Before spending GPU time on your own run, load it and confirm it actually detects PPE — a fast, motivating check that the whole pipeline (weights → inference → drawn boxes) works end to end.


In [ ]:
from ultralytics import YOLO
import random

REFERENCE_WEIGHTS = DATA_DIR.parent / "results_yolov8n_100e" / "kaggle" / "working" / "runs" / "detect" / "train" / "weights" / "best.pt"

reference_model = YOLO(str(REFERENCE_WEIGHTS))
sample_image = random.choice(list((DATA_DIR / "test" / "images").glob("*")))
results = reference_model(str(sample_image))
results[0].show()   # or results[0].save("sanity_check.jpg")


> **Worth being precise about:** loading `best.pt` above and running inference is *not* your team doing transfer learning — no training happens, so nothing is being transferred. It's just running someone else's already-fine-tuned model. The transfer learning step already happened, once, when the reference notebook's author fine-tuned `yolov8n.pt` (COCO-pretrained) on `css-data` — you're reusing their finished result as a sanity check / baseline to beat, not repeating that step yourselves. Section 5's training cells are where *you* do the actual transfer learning: starting from COCO weights and fine-tuning on this dataset yourselves, which is presumably the part of Phase 1 the pitch (and the bootcamp's Transfer Learning module) is actually asking you to demonstrate.


## 3 · Exploratory Data Analysis

Same instinct as `Module 22`: look at the structure of the data *before* trusting any model output. For a detection dataset that means, at minimum:

1. **Class balance** — per the pitch's own risk register, `NO-Hardhat` / `NO-Mask` / `NO-Safety Vest` are expected to be far rarer than their positive counterparts. Confirm it, don't assume it.
2. **Image properties** — resolution, aspect ratio, lighting variety (site photos won't be as clean as ImageNet).
3. **Box size distribution** — small/occluded objects (a bare head at a distance) are called out in the pitch as the hardest case; know how many of those you actually have.
4. **Visual sanity check** — draw the YOLO-format label boxes on a handful of images to catch annotation issues early.


In [ ]:
def parse_yolo_labels(label_path, class_names):
    """Read a YOLO-format .txt label file into a list of (class_name, x, y, w, h) — all normalised 0-1."""
    rows = []
    if not Path(label_path).exists():
        return rows
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            rows.append((class_names[cls_id], x, y, w, h))
    return rows


In [ ]:
# Class distribution across the training split
records = []
train_labels_dir = DATA_DIR / "train" / "labels"
if train_labels_dir.exists():
    for lbl_file in train_labels_dir.glob("*.txt"):
        for cls_name, x, y, w, h in parse_yolo_labels(lbl_file, CLASS_NAMES):
            records.append({"class": cls_name, "w": w, "h": h, "area": w * h})

df_labels = pd.DataFrame(records)

if not df_labels.empty:
    plt.figure(figsize=(9, 4))
    order = df_labels["class"].value_counts().index
    sns.countplot(data=df_labels, y="class", order=order)
    plt.title("Bounding box count per class (train split)")
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()
else:
    print("No labels parsed yet — download the dataset first.")


In [ ]:
# Box-area distribution (proxy for "how many small/occluded objects are we dealing with?")
if not df_labels.empty:
    plt.figure(figsize=(9, 4))
    sns.histplot(df_labels["area"], bins=40)
    plt.title("Bounding box area (normalised) distribution")
    plt.xlabel("box area (fraction of image)")
    plt.show()

    print(df_labels.groupby("class")["area"].median().sort_values())


In [ ]:
def draw_boxes(image_path, label_path, class_names):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img = img.shape[:2]

    for cls_name, x, y, w, h in parse_yolo_labels(label_path, class_names):
        x1, y1 = int((x - w / 2) * w_img), int((y - h / 2) * h_img)
        x2, y2 = int((x + w / 2) * w_img), int((y + h / 2) * h_img)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, cls_name, (x1, max(y1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return img


# Sanity-check a handful of random training images with their labels overlaid
train_images_dir = DATA_DIR / "train" / "images"
if train_images_dir.exists():
    sample_images = random.sample(list(train_images_dir.glob("*")), k=min(6, len(list(train_images_dir.glob("*")))))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample_images):
        lbl_path = train_labels_dir / (img_path.stem + ".txt")
        ax.imshow(draw_boxes(img_path, lbl_path, CLASS_NAMES))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
